In [1]:

# First Order Logic Resolution
# Problem: Prove that John likes peanuts

def is_variable(x):
    return x.startswith("?")

def substitute(term, subst):
    while term in subst:
        term = subst[term]
    return term

def unify(args1, args2, subst):
    subst = subst.copy()

    for a, b in zip(args1, args2):
        a = substitute(a, subst)
        b = substitute(b, subst)

        if a == b:
            continue

        if is_variable(a):
            subst[a] = b
        elif is_variable(b):
            subst[b] = a
        else:
            return None

    return subst

def apply_subst(literal, subst):
    sign, pred, args = literal
    return (sign, pred,
            tuple(substitute(x, subst) for x in args))

def standardize(clause, suffix):
    mapping = {}

    result = []

    for sign, pred, args in clause:
        new_args = []

        for arg in args:
            if is_variable(arg):
                if arg not in mapping:
                    mapping[arg] = arg + str(suffix)
                new_args.append(mapping[arg])
            else:
                new_args.append(arg)

        result.append((sign, pred, tuple(new_args)))

    return result

def canonical(clause):
    mapping = {}
    result = []

    for sign, pred, args in clause:
        new_args = []

        for arg in args:
            if is_variable(arg):
                if arg not in mapping:
                    mapping[arg] = "?V" + str(len(mapping))
                new_args.append(mapping[arg])
            else:
                new_args.append(arg)

        result.append((sign, pred, tuple(new_args)))

    return tuple(sorted(result))

def resolve(c1, c2, suffix):
    c1 = standardize(c1, "a" + str(suffix))
    c2 = standardize(c2, "b" + str(suffix))

    resolvents = []

    for i, l1 in enumerate(c1):
        for j, l2 in enumerate(c2):

            sign1, pred1, args1 = l1
            sign2, pred2, args2 = l2

            if pred1 != pred2 or sign1 == sign2:
                continue

            subst = unify(args1, args2, {})

            if subst is None:
                continue

            new_clause = []

            for k, lit in enumerate(c1):
                if k != i:
                    new_clause.append(apply_subst(lit, subst))

            for k, lit in enumerate(c2):
                if k != j:
                    new_clause.append(apply_subst(lit, subst))

            # Remove duplicate literals
            new_clause = list(set(new_clause))

            # Ignore tautological clauses
            if any(
                (not sign, pred, args) in new_clause
                for sign, pred, args in new_clause
            ):
                continue

            resolvents.append(new_clause)

    return resolvents

def display(clause):
    if not clause:
        return "EMPTY CLAUSE (Contradiction)"

    parts = []

    for sign, pred, args in clause:
        atom = pred + "(" + ", ".join(args) + ")"

        if not sign:
            atom = "~" + atom

        parts.append(atom)

    return " OR ".join(parts)

# Knowledge Base in CNF

KB = [

    # John likes all kinds of food
    [(False, "Food", ("?x",)),
     (True, "Likes", ("John", "?x"))],

    # Apple and vegetables are food
    [(True, "Food", ("Apple",))],

    [(True, "Food", ("Vegetables",))],

    # Anything eaten by anyone who is not killed is food
    [(False, "Eats", ("?x", "?y")),
     (False, "NotKilled", ("?x",)),
     (True, "Food", ("?y",))],

    # Anil eats peanuts
    [(True, "Eats", ("Anil", "Peanuts"))],

    # Anil is alive
    [(True, "Alive", ("Anil",))],

    # Anyone alive implies not killed
    [(False, "Alive", ("?x",)),
     (True, "NotKilled", ("?x",))],

    # Anyone not killed implies alive
    [(False, "NotKilled", ("?x",)),
     (True, "Alive", ("?x",))],

    # Harry eats everything Anil eats
    [(False, "Eats", ("Anil", "?x")),
     (True, "Eats", ("Harry", "?x"))]
]

# Negation of the goal
# Goal: Likes(John, Peanuts)

goal = [
    [(False, "Likes", ("John", "Peanuts"))]
]

# Resolution algorithm

clauses = KB + goal
support = [len(clauses) - 1]

seen = {canonical(c) for c in clauses}

print("Initial Knowledge Base:")
for i, clause in enumerate(clauses):
    print("C" + str(i + 1), ":", display(clause))

print("\nStarting Resolution...\n")

step = 0
proved = False

while support and step < 1000:

    new_support = []

    for i in support:
        for j in range(len(clauses)):

            if i == j:
                continue

            results = resolve(clauses[i], clauses[j], step)

            for result in results:

                key = canonical(result)

                if key in seen:
                    continue

                seen.add(key)

                step += 1

                print("Step", step, ":",
                      display(clauses[i]), "+",
                      display(clauses[j]))

                print("       =>", display(result))
                print()

                if not result:
                    proved = True
                    break

                clauses.append(result)
                new_support.append(len(clauses) - 1)

            if proved:
                break

        if proved:
            break

    if proved:
        break

    support = new_support

if proved:
    print("SUCCESS!")
    print("Contradiction obtained: Empty Clause")
    print("Therefore, John likes peanuts is PROVED.")
else:
    print("Proof not found within the search limit.")

Initial Knowledge Base:
C1 : ~Food(?x) OR Likes(John, ?x)
C2 : Food(Apple)
C3 : Food(Vegetables)
C4 : ~Eats(?x, ?y) OR ~NotKilled(?x) OR Food(?y)
C5 : Eats(Anil, Peanuts)
C6 : Alive(Anil)
C7 : ~Alive(?x) OR NotKilled(?x)
C8 : ~NotKilled(?x) OR Alive(?x)
C9 : ~Eats(Anil, ?x) OR Eats(Harry, ?x)
C10 : ~Likes(John, Peanuts)

Starting Resolution...

Step 1 : ~Likes(John, Peanuts) + ~Food(?x) OR Likes(John, ?x)
       => ~Food(Peanuts)

Step 2 : ~Food(Peanuts) + ~Eats(?x, ?y) OR ~NotKilled(?x) OR Food(?y)
       => ~NotKilled(?xb1) OR ~Eats(?xb1, Peanuts)

Step 3 : ~NotKilled(?xb1) OR ~Eats(?xb1, Peanuts) + Eats(Anil, Peanuts)
       => ~NotKilled(Anil)

Step 4 : ~NotKilled(?xb1) OR ~Eats(?xb1, Peanuts) + ~Alive(?x) OR NotKilled(?x)
       => ~Eats(?xb3, Peanuts) OR ~Alive(?xb3)

Step 5 : ~NotKilled(?xb1) OR ~Eats(?xb1, Peanuts) + ~Eats(Anil, ?x) OR Eats(Harry, ?x)
       => ~NotKilled(Harry) OR ~Eats(Anil, Peanuts)

Step 6 : ~NotKilled(Anil) + ~Alive(?x) OR NotKilled(?x)
       => ~Alive(An